# 04 - Tools Layer

This notebook defines the reusable tools used by the agent:
- search_runbooks
- generate_validation_sql
- build_grounded_prompt
- summarize_recovery_steps
- route_question

In [0]:
%pip install -q databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient
from typing import List, Dict

In [0]:
# -----------------------------
# Configuration
# -----------------------------
VECTOR_SEARCH_ENDPOINT = "rag-demo-endpoint"
VECTOR_INDEX_NAME = "workspace.rag_demo.rag_chunks_index"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

client = VectorSearchClient(disable_notice=True)
index = client.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME
)

In [0]:
def search_runbooks(question: str, num_results: int = 3) -> List[str]:
    """
    Retrieve top-matching runbook chunks from Vector Search.
    """
    results = index.similarity_search(
        query_text=question,
        columns=["chunk"],
        num_results=num_results
    )

    return [row[0] for row in results["result"]["data_array"]]

In [0]:
def generate_validation_sql(issue_type: str) -> str:
    """
    Return canned SQL templates based on issue type.
    """
    issue_type = issue_type.lower()

    if "duplicate" in issue_type:
        return """
SELECT surrogate_key, COUNT(*)
FROM target_table
GROUP BY surrogate_key
HAVING COUNT(*) > 1;
""".strip()

    if "null" in issue_type:
        return """
SELECT *
FROM target_table
WHERE important_column IS NULL;
""".strip()

    if "row count" in issue_type or "count" in issue_type:
        return """
SELECT COUNT(*) AS source_count FROM source_table;

SELECT COUNT(*) AS target_count FROM target_table;
""".strip()

    if "consistency" in issue_type or "compare" in issue_type:
        return """
SELECT *
FROM source_table
EXCEPT
SELECT *
FROM target_table;
""".strip()

    return """
-- No predefined SQL template matched the issue type.
-- Consider using one of these issue types:
-- duplicate, null, row count, consistency
""".strip()

In [0]:
def build_grounded_prompt(question: str, chunks: List[str]) -> str:
    """
    Create a grounded prompt using retrieved chunks.
    """
    context = "\n\n".join(chunks)

    return f"""
You are a senior data engineering troubleshooting assistant.

Use ONLY the context below.
Do not invent facts that are not present in the context.

Context:
{context}

Question:
{question}

Return in this format:
1. Likely cause
2. Recommended steps
3. Validation steps
"""

In [0]:
def summarize_recovery_steps(question: str, chunks: List[str]):
    """
    Use ai_query to synthesize a grounded answer from retrieved context.
    """
    prompt = build_grounded_prompt(question, chunks)
    safe_prompt = prompt.replace("'", "''")

    return spark.sql(f"""
    SELECT ai_query(
      '{LLM_ENDPOINT}',
      '{safe_prompt}'
    ) AS answer
    """)

In [0]:
def route_question(question: str) -> Dict[str, str]:
    """
    Simple deterministic router for lightweight agentic behavior.
    """
    q = question.lower()

    if any(term in q for term in ["sql", "query", "duplicate", "null", "row count", "count", "consistency"]):
        return {"tool": "generate_validation_sql", "reason": "Question appears to ask for validation SQL."}

    if any(term in q for term in ["fix", "failed", "failure", "troubleshoot", "error", "rerun", "recover"]):
        return {"tool": "search_and_summarize", "reason": "Question appears to ask for troubleshooting guidance."}

    if any(term in q for term in ["summary", "summarize", "what should i do", "next steps"]):
        return {"tool": "search_and_summarize", "reason": "Question appears to ask for action guidance."}

    return {"tool": "search_and_summarize", "reason": "Defaulting to grounded retrieval and synthesis."}

In [0]:
def print_chunks(chunks: List[str]) -> None:
    """
    Pretty-print retrieved chunks for debugging/demo.
    """
    for i, chunk in enumerate(chunks, start=1):
        print("\n" + "=" * 100)
        print(f"Chunk {i}")
        print("=" * 100)
        print(chunk[:1500])